# Figure 1.4: Random walks on a cyclic group

*The Discrete Algebra of the Fourier Transform*, Gabriel Peyré.

Random walks on $\mathbb{Z}/15\mathbb{Z}$ with increment laws $(0,1/2,0,\ldots,0,1/2)$ and $(0.3,0.3,0.2,0,\ldots,0,0.1,0.1)$. Dashed lines show the uniform distribution.

[Read this figure in the book](https://mathematical-tours.github.io/daft-sources/DAFT-EN.pdf#page=19).

## Prepare the example

Run the cells in order in Jupyter or Google Colab. The first cell installs missing Python packages and downloads the figure sources and input data once. Subsequent runs use the cached download. Python 3.10 or later is required.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import io
import os
import subprocess
import sys
import urllib.request
import zipfile

packages = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
            "PIL": "Pillow", "sympy": "sympy"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

archive_url = os.environ.get("DAFT_FIGURE_ARCHIVE", 'https://mathematical-tours.github.io/daft-sources/downloads/daft-figures-2e843bbe89f0.zip')
archive_sha = '2e843bbe89f0e274849b6896eb16bd9f7686521a8ca98f633240bfc06ea2d5b1'
workspace = Path(os.environ.get("DAFT_FIGURE_WORKDIR", "daft-figure-files")) / archive_sha[:12]
cache = workspace / "sources.zip"
workspace.mkdir(parents=True, exist_ok=True)
data = cache.read_bytes() if cache.exists() else urllib.request.urlopen(archive_url, timeout=60).read()
if hashlib.sha256(data).hexdigest() != archive_sha:
    raise ValueError("The figure-source download is incomplete or has changed. Download it again.")
cache.write_bytes(data)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    archive.extractall(workspace)
ROOT = (workspace / "daft-figures").resolve()
__file__ = str(ROOT / 'figures-code/chapter-finite-groups/random-walks/generate.py')
print("Figure sources ready.")


## Generate the figure

This is the plotting code used by the book. Change its parameters and rerun this cell, then the display cell below, to explore the example.

In [ ]:
"""Two probability kernels on Z/15Z, with the same four observation times."""
from pathlib import Path
import sys
sys.path.insert(0, str(Path(__file__).resolve().parents[3] / 'tools'))
from figure_style import *

n=15
v1=np.zeros(n); v1[[1,-1]]=.5
v2=np.zeros(n); v2[[0,1,2,-2,-1]]=[.3,.3,.2,.1,.1]
fig,axs=plt.subplots(4,2,figsize=(7.2,5.0),sharex=True,sharey='row')
for col,v in enumerate([v1,v2]):
    transition=np.array([[v[(j-i)%n] for i in range(n)] for j in range(n)])
    for row,k in enumerate([0,1,5,20]):
        p=np.fft.ifft(np.fft.fft(v)**k).real
        assert np.allclose(p,np.linalg.matrix_power(transition,k)[:,0])
        assert np.isclose(p.sum(),1) and p.min()>-1e-12
        ax=axs[row,col]
        discrete(ax,np.arange(n),p,TEAL if col==0 else BLUE)
        ax.axhline(1/n,color=RED,lw=.8,ls='--')
        ax.text(.5,.82,rf'$k={k}$',ha='center',transform=ax.transAxes)
        ax.set(xlim=(-.4,14.4),xticks=[0,5,10,14])
        axes_box(ax,'$x$' if row==3 else None,'$p^{(k)}(x)$' if col==0 else None)
        if row==0:
            ax.set_title('Nearest-neighbor walk' if col==0 else 'Asymmetric walk')
axs[3,1].text(.98,.55,'Uniform law',color=RED,fontsize=8,ha='right',transform=axs[3,1].transAxes)
save(fig,'chapter-finite-groups','random-walks')


## Display and save

In [ ]:
from IPython.display import Image, FileLink, display

display(Image(filename=str(ROOT / 'build/figure-previews/chapter-finite-groups/random-walks.png')))
display(FileLink(os.path.relpath(ROOT / 'figures/chapter-finite-groups/random-walks/figure.pdf', Path.cwd()),
                 result_html_prefix="Download the vector PDF: "))
